In [ ]:
import os

TAG = "PS_ConvLSTMPlus"

RUN_MODE  = "train"            # "check" | "train" | "evaluate"
SEED      = 0
OVERWRITE = False

INPUT_SET   = "fnoplus"
USE_PHYSICS = True
HIDDEN = 64
MODEL_FIELDS = {"arch": "ConvLSTM", "hidden": HIDDEN, "kernel": 3, "norm": "none", "head_width": 32,
                "input_set": INPUT_SET}

REGION = "Pakistan"            # "Pakistan" | "Australia60"
TRANSFER_TO = None             # None = the paper's targets for REGION

PAPER_EPOCHS = 100
CROP_EPOCHS  = 300
EPOCHS       = None            # an int overrides PAPER_EPOCHS / CROP_EPOCHS
BUDGET       = "timebox"       # "timebox" caps crop training at CROP_EPOCHS | "paper" matches 100 full-domain passes
BATCH_SIZE   = 1
LR           = 1e-3
WEIGHT_DECAY = 1e-4

TRAIN_PATCH = 128              # None = full-domain training
TRAIN_PATCH_MIN_CELLS = 128 * 128 * 4

VAL_EVERY   = 5
VAL_SAMPLES = 8
EVAL_DEVICE    = "auto"        # "auto" | "cuda" | "cpu"
EVAL_VRAM_FRAC = 0.80

TRANSFER_RENORM = False        # False = frozen weights with the source normalisation, as in the paper

RUN_MODE = os.environ.get("RUN_MODE", RUN_MODE)
REGION   = os.environ.get("REGION", REGION)
SEED     = int(os.environ.get("SEED", SEED))
EPOCHS   = int(os.environ["EPOCHS"]) if os.environ.get("EPOCHS") else EPOCHS
BUDGET   = os.environ.get("BUDGET", BUDGET)
TRAIN_PATCH = int(os.environ["TRAIN_PATCH"]) if os.environ.get("TRAIN_PATCH") else TRAIN_PATCH

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("rasterio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "rasterio"], check=True)

# Set before rasterio is imported: skips GDAL's per-open directory listing, which is O(N^2) over a frame folder.
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
os.environ.setdefault("GDAL_NUM_THREADS", "ALL_CPUS")

import hashlib
import json
import math
import random
import time
import warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import rasterio
from rasterio.errors import NotGeoreferencedWarning
from scipy import ndimage
from scipy.interpolate import RegularGridInterpolator

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

SESSION_T0 = time.time()
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VRAM_BYTES = torch.cuda.get_device_properties(0).total_memory if DEVICE.type == "cuda" else 0


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)


def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return None


def fmt_hms(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}"


print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", np.__version__)
if DEVICE.type == "cuda":
    _gpu = torch.cuda.get_device_properties(0)
    print(f"gpu: {_gpu.name}  {_gpu.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no CUDA GPU found - training runs on the CPU and is 10-50x slower.")

In [ ]:
DATASET_DIRNAME = "FloodCastBench"
DATASET_SUBDIRS = ("Relevant data", "Low-fidelity flood forecasting", "High-fidelity flood forecasting")


def find_data_root() -> str:
    candidates, here = [], os.path.abspath(os.getcwd())
    while True:
        candidates.append(os.path.join(here, DATASET_DIRNAME))
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    for candidate in candidates:
        if all(os.path.isdir(os.path.join(candidate, d)) for d in DATASET_SUBDIRS):
            return candidate
    raise FileNotFoundError(
        f"FloodCastBench not found. Expected Local Results/{DATASET_DIRNAME}/ holding "
        f"{', '.join(repr(d) for d in DATASET_SUBDIRS)}. Looked at:\n  " + "\n  ".join(candidates[:8]))


DATA_ROOT   = find_data_root()
LOCAL_BASE  = os.path.dirname(DATA_ROOT)
CACHE_DIR   = os.path.join(LOCAL_BASE, "cache")
# File names do not carry the patch size, so a non-128 patch gets its own folder.
RESULTS_DIR = os.path.join(LOCAL_BASE, "results_final",
                           "paper_standard" + ("" if TRAIN_PATCH == 128 else f"_p{TRAIN_PATCH}"))
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

BLOCK     = 20
OUT_STEPS = BLOCK - 1
PAPER_SAMPLE_COUNTS = {"Pakistan": 201, "Mozambique": 86, "Australia60": 144, "Australia30": 144,
                       "UK60": 43, "UK30": 43}

PAPER_SPLITS = {
    "Pakistan":    (145, 28, 28),
    "Australia60": (116, 14, 14),
    "Australia30": (116, 14, 14),
}
TRANSFER_PRESETS = {
    "Pakistan":    ("Mozambique",),
    "Australia60": ("UK60", "UK30"),
    "Australia30": ("UK30",),
}
TRANSFER = tuple(TRANSFER_TO) if TRANSFER_TO else TRANSFER_PRESETS.get(REGION, ())


@dataclass
class Config:
    tag: str = TAG
    region: str = REGION
    seed: int = SEED
    block: int = BLOCK
    out_steps: int = OUT_STEPS
    use_physics: bool = USE_PHYSICS
    target_transform: str = "linear"
    clamp_output: bool = False
    report_weights: str = "final"
    budget: str = BUDGET
    loss_space: str = "metres"     # "metres" | "normalised"
    residual: bool = False
    epochs: int = 0
    batch_size: int = BATCH_SIZE
    lr: float = LR
    weight_decay: float = WEIGHT_DECAY
    train_patch: int = 0
    grad_clip: float = 0.0
    model: tuple = ()


PIPELINE_VERSION = "PS1"

CFG = Config(model=tuple(sorted(MODEL_FIELDS.items())))
set_seed(CFG.seed)


def experiment_key(cfg) -> dict:
    key = json.loads(json.dumps(asdict(cfg), default=str))
    key["pipeline"] = PIPELINE_VERSION
    return key


RUN_NAME    = f"{TAG}_{REGION}_seed{SEED}"
RESULT_JSON = os.path.join(RESULTS_DIR, f"{RUN_NAME}.json")
CKPT_PATH   = os.path.join(RESULTS_DIR, f"{RUN_NAME}.pt")

if RUN_MODE not in ("check", "train", "evaluate"):
    raise ValueError(f"RUN_MODE={RUN_MODE!r}; choose from ('check', 'train', 'evaluate')")
if REGION not in PAPER_SPLITS:
    print(f"NOTE: {REGION} has no paper split; falling back to a 72/14/14 chronological split.")

print(f"RUN_MODE={RUN_MODE} | {TAG} | train on {REGION} | seed {SEED} | device {DEVICE}")
print(f"transfer targets, frozen weights: {TRANSFER if TRANSFER else '(none)'}")
print(f"task: non-overlapping blocks of {BLOCK} frames x 300 s; predict t=2..{BLOCK} from t=1 "
      f"({OUT_STEPS} steps = {OUT_STEPS * 300 / 60:.0f} min lead)")
print(f"inputs: X, Y, T, depth(t=1)" + (" + DEM + rainfall(t=1..20)" if USE_PHYSICS else ""))
print("data_root:", DATA_ROOT)
print("cache_dir:", CACHE_DIR, "| save_dir:", RESULTS_DIR)

In [ ]:
GRID_SPEC = {
    "Mozambique":  dict(crop=(150, 50, 50, 50), factor=16, shape=(151, 138),
                        flood=("Low-fidelity flood forecasting", "480m"), dem="Mozambique_DEM.tif",
                        lulc="Mozambique.tif", rain="Mozambique flood", cell_m=485.0),
    "Australia60": dict(crop=(0, 0, 0, 0), factor=2, shape=(536, 536),
                        flood=("High-fidelity flood forecasting", "60m"), dem="Australia_DEM.tif",
                        lulc="Australia.tif", rain="Australia flood", region_dir="Australia", cell_m=60.0),
    "Australia30": dict(crop=(0, 0, 0, 0), factor=1, shape=(1073, 1073),
                        flood=("High-fidelity flood forecasting", "30m"), dem="Australia_DEM.tif",
                        lulc="Australia.tif", rain="Australia flood", region_dir="Australia", cell_m=30.0),
    "UK60":        dict(crop=(5, 5, 5, 5), factor=2, shape=(85, 137),
                        flood=("High-fidelity flood forecasting", "60m"), dem="UK_DEM.tif",
                        lulc="UK.tif", rain="UK flood", region_dir="UK", cell_m=60.0),
    "UK30":        dict(crop=(5, 5, 5, 5), factor=1, shape=(170, 275),
                        flood=("High-fidelity flood forecasting", "30m"), dem="UK_DEM.tif",
                        lulc="UK.tif", rain="UK flood", region_dir="UK", cell_m=30.0),
    # Pakistan's crop is not stated by FloodCastBench; it was fitted against corr(DEM, depth).
    "Pakistan":    dict(crop=(40, 35, 182, 60), factor=16, shape=(810, 441),
                        flood=("Low-fidelity flood forecasting", "480m"), dem="Pakistan_DEM.tif",
                        lulc="Pakistan.tif", rain="Pakistan flood", provisional=True, cell_m=485.0),
}

MANNING_LUT = {0: 0.0375, 1: 0.0350, 2: 0.1200, 4: 0.0800, 5: 0.0350,
               7: 0.3750, 8: 0.0265, 9: 0.0375, 10: 0.0375, 11: 0.0375}
NODATA = -2000.0

# Only elevation is used, but the cache key hashes all eleven channels to stay shared with the M-series cache.
STATIC_FULL = ("elevation", "slope", "aspect_sin", "aspect_cos", "curvature",
               "relief_1km", "relief_5km", "sink_depth", "manning", "water_frac", "built_frac")


def inter(a, size):
    a = np.asarray(a, dtype=np.float64)
    h, w = a.shape
    if size == 1:
        return a.copy()
    nh, nw = h // size, w // size
    f = RegularGridInterpolator((np.arange(h, dtype=float), np.arange(w, dtype=float)), a,
                                method="linear", bounds_error=False, fill_value=None)
    Y, X = np.meshgrid(np.linspace(0, h - 1, nh), np.linspace(0, w - 1, nw), indexing="ij")
    return f(np.stack([Y, X], axis=-1))


def crop30(a, spec):
    t, b, l, r = spec["crop"]
    return a[t: a.shape[0] - b if b else None, l: a.shape[1] - r if r else None]


def read_tif(path):
    with rasterio.open(path) as src:
        return src.read(1)


def fix_missing_values(a):
    a = np.asarray(a, dtype=np.float64).copy()
    missing = ~np.isfinite(a) | (a < NODATA)
    if missing.any():
        a[missing] = 30.0 + a[~missing].max()
    return a


def bilinear_taps(n, size):
    x = np.linspace(0, n - 1, n // size)
    i0 = np.minimum(np.floor(x).astype(np.int64), n - 2)
    return i0, x - i0


def inter_rain(a, size):
    if size == 1:
        g = np.asarray(a, dtype=np.float64).copy()
        g[g < 0] = 0.0
        return g
    (r0, fy), (c0, fx) = bilinear_taps(a.shape[0], size), bilinear_taps(a.shape[1], size)
    nh, nw = len(r0), len(c0)
    g = a[np.ix_(np.concatenate([r0, r0 + 1]), np.concatenate([c0, c0 + 1]))].astype(np.float64)
    g[g < 0] = 0.0
    fy, fx = fy[:, None], fx[None, :]
    return (g[:nh, :nw] * (1 - fy) * (1 - fx) + g[:nh, nw:] * (1 - fy) * fx
            + g[nh:, :nw] * fy * (1 - fx) + g[nh:, nw:] * fy * fx)


print("grid specs loaded for:", ", ".join(GRID_SPEC))

In [ ]:
REGIONS = {}


def cache_tag(region, spec):
    return (f"{region}_" + hashlib.md5(json.dumps(
        [region, list(spec["crop"]), spec["factor"], list(STATIC_FULL)]).encode()).hexdigest()[:10])


def load_region(region, verbose=True):
    if region in REGIONS:
        return REGIONS[region]
    if region not in GRID_SPEC:
        raise KeyError(f"unknown region {region!r}; this dataset holds {list(GRID_SPEC)}")
    spec = GRID_SPEC[region]
    H, W = spec["shape"]
    tag  = cache_tag(region, spec)
    p_depth  = os.path.join(CACHE_DIR, f"depth_{tag}.npy")
    p_rain   = os.path.join(CACHE_DIR, f"rain_{tag}.npy")
    p_static = os.path.join(CACHE_DIR, f"static_{tag}.npz")
    p_man    = os.path.join(CACHE_DIR, f"manifest_{tag}.json")

    rel  = os.path.join(DATA_ROOT, "Relevant data")
    fdir = os.path.join(DATA_ROOT, spec["flood"][0], spec["flood"][1], spec.get("region_dir", region))

    def build_depth():
        times = sorted(int(f[:-4]) for f in os.listdir(fdir) if f.endswith(".tif"))
        dt = times[1] - times[0]
        assert all(t == i * dt for i, t in enumerate(times)), "non-uniform time axis"
        out = np.lib.format.open_memmap(p_depth, mode="w+", dtype=np.float32, shape=(len(times), H, W))
        every, t0 = max(1, len(times) // 10), time.time()
        for i, t in enumerate(times):
            a = read_tif(os.path.join(fdir, f"{t}.tif"))
            assert a.shape == (H, W), f"frame {t}: {a.shape} != {(H, W)}"
            out[i] = np.maximum(np.nan_to_num(a, nan=0.0), 0.0)
            if (i + 1) % every == 0:
                print(f"    depth {i + 1}/{len(times)}  {time.time() - t0:.0f}s", flush=True)
        out.flush()
        n = int(out.shape[0])
        del out
        return n, dt

    def build_rain():
        rdir = os.path.join(rel, "Rainfall", spec["rain"])
        files = sorted(f for f in os.listdir(rdir) if f.endswith(".tif"))
        out = np.empty((len(files), H, W), np.float32)
        every, t0 = max(1, len(files) // 10), time.time()
        for i, f in enumerate(files):
            out[i] = inter_rain(crop30(read_tif(os.path.join(rdir, f)), spec), spec["factor"]).astype(np.float32)
            if (i + 1) % every == 0:
                print(f"    rain {i + 1}/{len(files)}  {time.time() - t0:.0f}s", flush=True)
        return out, files

    def build_static():
        dem_raw  = read_tif(os.path.join(rel, "DEM", spec["dem"]))
        lulc_raw = read_tif(os.path.join(rel, "Land use and land cover", spec["lulc"]))
        dem30  = crop30(fix_missing_values(dem_raw), spec)
        lulc30 = crop30(lulc_raw, spec).astype(np.uint8)
        px = 30.0
        w1 = int(round(1000 / px)) | 1
        w5 = int(round(5000 / px)) | 1
        lut = np.full(256, 0.0375)
        for k, v in MANNING_LUT.items():
            lut[k] = v
        ds = lambda a: inter(a, spec["factor"])
        gy, gx = np.gradient(dem30, px, px)
        ch = {"elevation": ds(dem30), "slope": ds(np.sqrt(gx ** 2 + gy ** 2))}
        aspect = np.arctan2(-gx, gy)
        del gx, gy
        ch["aspect_sin"], ch["aspect_cos"] = ds(np.sin(aspect)), ds(np.cos(aspect))
        del aspect
        ch["curvature"]  = ds(ndimage.laplace(dem30) / (px ** 2))
        ch["relief_1km"] = ds(dem30 - ndimage.uniform_filter(dem30, w1))
        ch["relief_5km"] = ds(dem30 - ndimage.uniform_filter(dem30, w5))
        ch["sink_depth"] = ds(dem30 - ndimage.minimum_filter(dem30, w1))
        ch["manning"]    = ds(lut[lulc30])
        ch["water_frac"] = ds((lulc30 == 1).astype(np.float64))
        ch["built_frac"] = ds((lulc30 == 7).astype(np.float64))
        for k, v in ch.items():
            assert v.shape == (H, W), f"static '{k}' is {v.shape}, expected {(H, W)}"
        out = {k: v.astype(np.float32) for k, v in ch.items()}
        out["bug_elevation"] = np.maximum(dem_raw[:H, :W].astype(np.float32), 0.0)
        out["bug_manning"]   = lulc_raw[:H, :W].astype(np.float32)
        return out

    if all(os.path.exists(p) for p in (p_depth, p_rain, p_static, p_man)):
        manifest = load_json(p_man)
        dt = manifest["dt_seconds"]
        rain = np.load(p_rain)
        static = dict(np.load(p_static))
        if verbose:
            print(f"{region}: cache hit")
    else:
        if verbose:
            print(f"{region}: building cache in {CACHE_DIR} (once; Pakistan ~6 min, the others under a minute)")
        t0 = time.time()
        n_frames, dt = build_depth()
        print(f"  depth  ({n_frames}, {H}, {W})  {time.time() - t0:.0f}s")
        rain, rain_files = build_rain()
        print(f"  rain   {rain.shape}  {time.time() - t0:.0f}s")
        static = build_static()
        print(f"  static {len(static)} arrays  {time.time() - t0:.0f}s")
        np.save(p_rain, rain)
        np.savez(p_static, **static)
        # The manifest is written last, so a half-written cache is rebuilt rather than trusted.
        save_json(p_man, dict(region=region, crop=list(spec["crop"]), factor=spec["factor"], shape=[H, W],
                              n_frames=int(n_frames), dt_seconds=int(dt), n_rain=int(rain.shape[0]),
                              rain_first=rain_files[0], rain_last=rain_files[-1],
                              static_channels=[k for k in static if not k.startswith("bug_")],
                              built=time.strftime("%Y-%m-%d %H:%M:%S")))

    depth = np.load(p_depth, mmap_mode="r")
    nt = depth.shape[0]
    yy = np.linspace(0.0, 1.0, H, dtype=np.float32)[:, None] * np.ones((1, W), np.float32)
    xx = np.ones((H, 1), np.float32) * np.linspace(0.0, 1.0, W, dtype=np.float32)[None, :]
    ctx = dict(region=region, spec=spec, H=H, W=W, depth=depth, rain=rain, dem=static["elevation"],
               dt=dt, nt=nt, X=xx, Y=yy, n_blocks=nt // BLOCK)
    if verbose:
        print(f"  {region}: depth {depth.shape} dt={dt}s span={nt * dt / 86400:.2f} d | rain {rain.shape} "
              f"| {nt // BLOCK} blocks of {BLOCK} frames | cell ~{spec['cell_m']:.0f} m")
    REGIONS[region] = ctx
    return ctx


CTX = load_region(REGION)
if CTX["dt"] != 300:
    raise RuntimeError(f"{REGION} has dt={CTX['dt']}s; the paper protocol is defined on 300 s frames")

In [ ]:
CHANNELS = ["X", "Y", "T", "depth_t1"] + (["DEM", "rain_t", "rain_t1"] if CFG.use_physics else [])
IN_CH = len(CHANNELS)


def split_blocks(ctx):
    n = ctx["n_blocks"]
    expect = PAPER_SAMPLE_COUNTS.get(ctx["region"])
    assert expect is None or n == expect, (
        f"{ctx['region']}: {n} blocks of {BLOCK} frames, but the paper reports {expect} samples")
    if ctx["region"] in PAPER_SPLITS:
        n_tr, n_va, n_te = PAPER_SPLITS[ctx["region"]]
        assert n_tr + n_va + n_te == n, f"{ctx['region']}: paper split {n_tr}/{n_va}/{n_te} != {n} blocks"
    else:
        n_tr = int(round(n * 0.72)); n_va = int(round(n * 0.14)); n_te = n - n_tr - n_va
    idx = np.arange(n)
    return idx[:n_tr], idx[n_tr:n_tr + n_va], idx[n_tr + n_va:]


TRAIN_B, VAL_B, TEST_B = split_blocks(CTX)
print(f"{REGION}: {CTX['n_blocks']} samples -> train {len(TRAIN_B)} | val {len(VAL_B)} | test {len(TEST_B)}"
      f"   (paper: {PAPER_SPLITS.get(REGION, 'transfer target, no split')})")


def rain_index(ctx, frame):
    return min(int(frame * ctx["dt"] // 1800), ctx["rain"].shape[0] - 1)


def fit_norm(ctx, blocks):
    frames = np.concatenate([np.arange(b * BLOCK, (b + 1) * BLOCK) for b in blocks])
    s = s2 = n = 0.0
    for a in range(0, len(frames), 64):
        x = np.asarray(ctx["depth"][frames[a:a + 64]], np.float64)
        s += x.sum(); s2 += (x * x).sum(); n += x.size
    mu = s / n
    sd = math.sqrt(max(s2 / n - mu * mu, 1e-12))
    n_r = rain_index(ctx, int(frames[-1])) + 1
    rs = rs2 = rn = 0.0
    for a in range(0, n_r, 32):
        x = np.asarray(ctx["rain"][a:min(a + 32, n_r)], np.float64)
        rs += x.sum(); rs2 += (x * x).sum(); rn += x.size
    r_mu = rs / rn
    r_sd = math.sqrt(max(rs2 / rn - r_mu * r_mu, 1e-12))
    dem = np.asarray(ctx["dem"], np.float64)
    return {"depth": (float(mu), float(sd)),
            "rain":  (float(r_mu), float(max(r_sd, 1e-6))),
            "dem":   (float(dem.mean()), float(max(dem.std(), 1e-6)))}


NORM = fit_norm(CTX, TRAIN_B)
print(f"norm (train blocks only, LINEAR): depth mu={NORM['depth'][0]:.4f} m sd={NORM['depth'][1]:.4f} m | "
      f"rain mu={NORM['rain'][0]:.4f} sd={NORM['rain'][1]:.4f} mm/h | "
      f"DEM mu={NORM['dem'][0]:.1f} sd={NORM['dem'][1]:.1f}")


def encode_depth(d_m, norm):
    return (np.asarray(d_m, np.float32) - norm["depth"][0]) / norm["depth"][1]


def decode_depth_t(z, norm, wet=None):
    mu, sd = norm["depth"]
    d = z * sd + mu
    return d.clamp(min=0.0) if CFG.clamp_output else d


def make_sample(ctx, b, norm, rows=None, cols=None):
    r0, r1 = rows if rows else (0, ctx["H"])
    c0, c1 = cols if cols else (0, ctx["W"])
    f0 = b * BLOCK
    h1 = np.asarray(ctx["depth"][f0, r0:r1, c0:c1], np.float32)
    y  = np.asarray(ctx["depth"][f0 + 1:f0 + BLOCK, r0:r1, c0:c1], np.float32)
    h, w = h1.shape
    K = y.shape[0]
    t_ax = np.linspace(0.0, 1.0, K, dtype=np.float32)
    bc = lambda a: np.broadcast_to(a[:, :, None], (h, w, K))

    ch = [bc(ctx["X"][r0:r1, c0:c1]),
          bc(ctx["Y"][r0:r1, c0:c1]),
          np.broadcast_to(t_ax[None, None, :], (h, w, K)),
          bc(encode_depth(h1, norm))]
    if CFG.use_physics:
        mu_r, sd_r = norm["rain"]
        rz = lambda a: ((np.maximum(a, 0.0).astype(np.float32) - mu_r) / sd_r).astype(np.float32)
        dem_z = ((ctx["dem"][r0:r1, c0:c1] - norm["dem"][0]) / norm["dem"][1]).astype(np.float32)
        # rain_t is the rain applied during the step that produces target frame k, as in the solver.
        r_t = np.stack([ctx["rain"][rain_index(ctx, f0 + k - 1), r0:r1, c0:c1] for k in range(1, K + 1)], -1)
        ch += [bc(dem_z), rz(r_t), bc(rz(ctx["rain"][rain_index(ctx, f0), r0:r1, c0:c1]))]
    return np.stack(ch, -1).astype(np.float32), np.ascontiguousarray(y.transpose(1, 2, 0))


DOMAIN_CELLS = CTX["H"] * CTX["W"]
if TRAIN_PATCH and DOMAIN_CELLS > TRAIN_PATCH_MIN_CELLS:
    PATCH = int(min(TRAIN_PATCH, CTX["H"], CTX["W"]))
else:
    PATCH = 0
CELL_RATIO = (PATCH * PATCH) / float(DOMAIN_CELLS) if PATCH else 1.0
_PAPER_CROP_EPOCHS = math.ceil(PAPER_EPOCHS / CELL_RATIO)
RUN_EPOCHS = EPOCHS or (PAPER_EPOCHS if not PATCH else
                        (_PAPER_CROP_EPOCHS if BUDGET == "paper" else min(CROP_EPOCHS, _PAPER_CROP_EPOCHS)))
CFG = Config(**{**asdict(CFG), "train_patch": PATCH, "epochs": RUN_EPOCHS, "model": CFG.model})
print(f"training geometry: " + (f"random {PATCH}x{PATCH} crops of the {CTX['H']}x{CTX['W']} domain"
                                if PATCH else f"full domain {CTX['H']}x{CTX['W']}")
      + f"  |  every REPORTED number is full-domain")

EPOCH_EQUIV = CFG.epochs * CELL_RATIO
print(f"training budget  : {CFG.epochs} epochs x {len(TRAIN_B)} samples = {CFG.epochs * len(TRAIN_B):,} steps"
      f" (~{EPOCH_EQUIV:.0f} full-domain passes)")
if EPOCH_EQUIV < 0.5 * PAPER_EPOCHS:
    print(f"  NOTE: well under the paper's {PAPER_EPOCHS} full-domain passes; matching it would need "
          f"{int(PAPER_EPOCHS / max(CELL_RATIO, 1e-9)):,} epochs")


def sample_patch(ctx, rng):
    if not PATCH:
        return None, None

    # Start drawn from [-(PATCH-1), n-1] and clamped, so edge cells are cropped as often as interior ones.
    def start(n):
        return int(np.clip(rng.integers(-(PATCH - 1), n), 0, max(n - PATCH, 0)))

    r0, c0 = start(ctx["H"]), start(ctx["W"])
    return (r0, r0 + PATCH), (c0, c0 + PATCH)


def to_torch(x, y, device):
    return (torch.from_numpy(x).unsqueeze(0).to(device),
            torch.from_numpy(y).unsqueeze(0).to(device))


_x, _y = make_sample(CTX, int(TEST_B[0]), NORM, *sample_patch(CTX, np.random.default_rng(0)))
print(f"one sample: x {_x.shape} ({IN_CH} channels: {', '.join(CHANNELS)}) | y {_y.shape} "
      f"| {_x.nbytes / 1e6:.1f} MB + {_y.nbytes / 1e6:.1f} MB")
del _x, _y

In [ ]:
_d_last = np.log1p(np.asarray(CTX["depth"][-1]).ravel())
_r_ok   = float(np.corrcoef(CTX["dem"].ravel(), _d_last)[0, 1])
_rain_mm  = float(CTX["rain"].mean(axis=(1, 2)).sum() * 0.5)
_depth_mm = float(np.asarray(CTX["depth"][-1]).mean() * 1000)
_ratio = _depth_mm / max(_rain_mm, 1e-9)

print(f"[1] mass balance : rained {_rain_mm:7.1f} mm | stored {_depth_mm:7.1f} mm | ratio {_ratio:.2f}"
      f"  (~1.0 where rain is the only source; Pakistan has an inflow boundary, so >1)")
print(f"[2] registration : corr(DEM, log1p depth) = {_r_ok:+.4f}   (must be negative)")
print(f"[3] blocks       : {CTX['n_blocks']} x {BLOCK} frames = {CTX['n_blocks'] * BLOCK} of {CTX['nt']} frames")
assert _ratio > 0.5, "mass balance failed - grid mapping or rainfall units are wrong"
assert _r_ok < 0,    "registration failed - the DEM is not on the flood grid"
assert len(TRAIN_B) and len(VAL_B) and len(TEST_B), "empty split"
assert max(TRAIN_B) < min(VAL_B) and max(VAL_B) < min(TEST_B), "splits are not chronological or overlap"
print("\nall checks passed")

_fig, _ax = plt.subplots(1, 3, figsize=(15, 3.6))
_days = np.arange(CTX["nt"]) * CTX["dt"] / 86400
_mean = np.concatenate([np.asarray(CTX["depth"][a:a + 64]).mean(axis=(1, 2))
                        for a in range(0, CTX["nt"], 64)])
_ax[0].plot(_days, _mean)
for _b, _c, _lab in ((TRAIN_B, "tab:blue", "train"), (VAL_B, "tab:orange", "val"), (TEST_B, "tab:red", "test")):
    _ax[0].axvspan(_b[0] * BLOCK * CTX["dt"] / 86400, (_b[-1] + 1) * BLOCK * CTX["dt"] / 86400,
                   color=_c, alpha=0.15, label=_lab)
_ax[0].set(xlabel="days", ylabel="mean depth [m]", title=f"{REGION}: domain-mean depth and the paper split")
_ax[0].legend(fontsize=8)
_ax[1].plot(np.arange(CTX["rain"].shape[0]) * 0.5 / 24, CTX["rain"].mean(axis=(1, 2)))
_ax[1].set(xlabel="days", ylabel="mm/h", title="domain-mean rainfall")
_im = _ax[2].imshow(np.asarray(CTX["depth"][-1]), cmap="Blues")
_ax[2].set(title=f"final frame, depth [m]"); _ax[2].axis("off")
plt.colorbar(_im, ax=_ax[2], fraction=0.046)
for _a in _ax[:2]:
    _a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
GAMMAS = (0.001, 0.01)


class PaperMetrics:
    def __init__(self, gammas=GAMMAS):
        self.g = tuple(gammas)
        self.n = self.sy = self.sy2 = self.sp = self.sp2 = self.syp = self.se = self.sae = 0.0
        self.tp = {g: 0.0 for g in self.g}; self.fp = dict(self.tp); self.fn = dict(self.tp)
        self.rel = []
        self.nse_s = []; self.r_s = []
        self.se_k = None; self.n_k = None

    @torch.no_grad()
    def update(self, pred_m, targ_m):
        p, y = pred_m.double(), targ_m.double()
        e = p - y
        self.n += y.numel()
        self.sy += y.sum().item(); self.sy2 += (y * y).sum().item()
        self.sp += p.sum().item(); self.sp2 += (p * p).sum().item(); self.syp += (p * y).sum().item()
        self.se += (e * e).sum().item(); self.sae += e.abs().sum().item()
        dims = tuple(range(1, y.ndim))
        self.rel += (torch.sqrt((e ** 2).sum(dim=dims)) /
                     (torch.sqrt((y ** 2).sum(dim=dims)) + 1e-12)).cpu().tolist()
        for i in range(y.shape[0]):
            yi, pi = y[i].flatten(), p[i].flatten()
            yc, pc = yi - yi.mean(), pi - pi.mean()
            sst = (yc * yc).sum().item()
            self.nse_s.append(1.0 - ((pi - yi) ** 2).sum().item() / max(sst, 1e-12))
            self.r_s.append((yc * pc).sum().item() / (math.sqrt(sst * (pc * pc).sum().item()) + 1e-12))
        per_k = (e ** 2).sum(dim=(0, 1, 2)).cpu().numpy()
        cnt_k = np.full(per_k.shape, y[..., 0].numel(), dtype=np.float64)
        self.se_k = per_k if self.se_k is None else self.se_k + per_k
        self.n_k  = cnt_k if self.n_k  is None else self.n_k + cnt_k
        for g in self.g:
            pm, ym = p > g, y > g
            self.tp[g] += (pm & ym).sum().item()
            self.fp[g] += (pm & ~ym).sum().item()
            self.fn[g] += (~pm & ym).sum().item()

    def result(self):
        n = max(self.n, 1.0)
        ybar, pbar = self.sy / n, self.sp / n
        sdy = math.sqrt(max(self.sy2 / n - ybar ** 2, 0.0))
        sdp = math.sqrt(max(self.sp2 / n - pbar ** 2, 0.0))
        # "RMSE" is the paper's Eq. 2: per-sample relative L2, not an error in metres (that is RMSE_m).
        out = {"RMSE": float(np.mean(self.rel)),
               "NSE": 1.0 - self.se / max(self.sy2 - n * ybar ** 2, 1e-12),
               "r": (self.syp / n - ybar * pbar) / (sdy * sdp + 1e-12),
               "RMSE_m": math.sqrt(self.se / n),
               "MAE_m": self.sae / n,
               "MSE_m2": self.se / n,
               "NSE_ps": float(np.mean(self.nse_s)),
               "r_ps": float(np.mean(self.r_s)),
               "n_samples": len(self.rel)}
        for g in self.g:
            tp, fp, fn = self.tp[g], self.fp[g], self.fn[g]
            out[f"CSI@{g}"] = tp / max(tp + fp + fn, 1e-12)
        out["_per_step_RMSE_m"] = np.sqrt(self.se_k / np.maximum(self.n_k, 1)).tolist()
        return out


PAPER_COLS = ("RMSE", "NSE", "r", "CSI@0.001", "CSI@0.01")
EXTRA_COLS = ("NSE_ps", "r_ps", "RMSE_m", "MAE_m", "n_samples")
LABELS = {"RMSE": "RMSE (rel-L2)", "NSE": "NSE", "r": "Pearson r", "CSI@0.001": "CSI g=0.001",
          "CSI@0.01": "CSI g=0.01", "NSE_ps": "NSE per-sample", "r_ps": "r per-sample", "RMSE_m": "RMSE [m]",
          "MAE_m": "MAE [m]", "n_samples": "samples"}
LOWER_IS_BETTER = {"RMSE", "RMSE_m", "MAE_m"}


def results_table(rows, title="", cols=PAPER_COLS + EXTRA_COLS, mark_best=True):
    if title:
        print(title)
    best = {}
    if mark_best:
        for c in cols:
            vals = {k: m[c] for k, m in rows.items() if c in m and np.isfinite(m[c]) and not k.startswith("paper")}
            if vals and c != "n_samples":
                best[c] = (min if c in LOWER_IS_BETTER else max)(vals, key=vals.get)
    print(f"  {'model':26s}" + "".join(f"{LABELS.get(c, c):>16s}" for c in cols))
    for name, m in rows.items():
        cells = ""
        for c in cols:
            v = m.get(c, float("nan"))
            s = f"{v:15.6f}" if isinstance(v, float) else f"{v:15}"
            cells += s + ("*" if best.get(c) == name else " ")
        print(f"  {name:26s}" + cells)


@torch.no_grad()
def persistence_reference(ctx, blocks, norm=None):
    acc = PaperMetrics()
    for b in blocks:
        f0 = int(b) * BLOCK
        y = torch.from_numpy(np.asarray(ctx["depth"][f0 + 1:f0 + BLOCK], np.float32)).permute(1, 2, 0)[None]
        p = torch.from_numpy(np.asarray(ctx["depth"][f0], np.float32))[None, :, :, None].expand_as(y)
        acc.update(p, y)
    return acc.result()


# Xu et al. (2025) Tables 4 and 5: RMSE, NSE, r, CSI@0.001, CSI@0.01.
PAPER_TABLE4 = {
    "Pakistan":    {"paper U-Net": (0.079399, 0.991705, 0.995934, 0.815218, 0.809986),
                    "paper FNO":   (0.002232, 0.999993, 0.999997, 0.966631, 0.992669),
                    "paper FNO+":  (0.002107, 0.999994, 0.999997, 0.976716, 0.993993)},
    "Australia60": {"paper U-Net": (0.566626, 0.560984, 0.942675, 0.779631, 0.837399),
                    "paper FNO":   (0.004258, 0.999975, 0.999987, 0.895553, 0.980748),
                    "paper FNO+":  (0.003941, 0.999979, 0.999990, 0.939638, 0.984588)},
}
PAPER_TABLE5 = {
    "Mozambique": {"paper U-Net": (1.602850, -72.350318, 0.903450, 0.724442, 0.667776),
                   "paper FNO":   (0.163300, 0.453167, 0.980383, 0.892473, 0.915509),
                   "paper FNO+":  (0.078633, 0.955450, 0.985521, 0.934028, 0.912712)},
    "UK60":       {"paper U-Net": (0.525467, 0.690026, 0.956584, 0.314035, 0.517440),
                   "paper FNO":   (0.028990, 0.998575, 0.999600, 0.755766, 0.925626),
                   "paper FNO+":  (0.024771, 0.998949, 0.999645, 0.859821, 0.963278)},
    "UK30":       {"paper U-Net": (0.556424, 0.660375, 0.960544, 0.282739, 0.553059),
                   "paper FNO":   (0.028297, 0.998706, 0.999604, 0.722694, 0.927164),
                   "paper FNO+":  (0.025190, 0.998953, 0.999639, 0.810674, 0.958525)},
}


def paper_rows(table, region):
    return {k: dict(zip(PAPER_COLS, v)) for k, v in table.get(region, {}).items()}


REF_TEST = persistence_reference(CTX, TEST_B)
REF_VAL  = persistence_reference(CTX, VAL_B)
results_table({**{"persistence (here)": REF_TEST}, **paper_rows(PAPER_TABLE4, REGION)},
              title=f"\n{REGION}: paper TEST split, {len(TEST_B)} samples, 19 steps = 95 min lead\n",
              mark_best=False)

In [ ]:
def make_norm(kind, ch):
    if kind == "batch":
        return nn.BatchNorm2d(ch)
    if kind == "group":
        return nn.GroupNorm(math.gcd(8, ch) or 1, ch)
    return nn.Identity()


class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid, k):
        super().__init__()
        self.hid = hid
        self.conv = nn.Conv2d(in_ch + hid, 4 * hid, k, padding=k // 2)
        nn.init.zeros_(self.conv.bias)
        self.conv.bias.data[hid:2 * hid] = 1.0              # forget-gate bias = 1

    def forward(self, x, state):
        h, c = state
        i, f, o, g = torch.split(self.conv(torch.cat([x, h], 1)), self.hid, dim=1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        return torch.sigmoid(o) * torch.tanh(c), c

    def init_state(self, b, hw, device, dtype):
        z = torch.zeros(b, self.hid, *hw, device=device, dtype=dtype)
        return z, z.clone()


class OutputHead(nn.Module):
    def __init__(self, in_ch, width=32, k=3):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_ch, width, k, padding=k // 2),
                                 nn.LeakyReLU(0.01, inplace=True),
                                 nn.Conv2d(width, 1, 1))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)


class RolloutConvLSTM(nn.Module):
    DEPTH_CH = 3          # CHANNELS = [X, Y, T, depth_t1, (DEM, rain_t, rain_t1)]
    T_CH     = 2
    RAIN_T   = 5
    RAIN_1   = 6

    def __init__(self, in_ch, hidden=64, kernel=3, norm="none", head_width=32, physics=False):
        super().__init__()
        self.physics = physics
        self.n_aux = in_ch - 1
        self.cell = ConvLSTMCell(1 + self.n_aux, hidden, kernel)
        self.mid  = make_norm(norm, hidden)
        self.head = OutputHead(hidden, head_width, kernel)

    def _aux(self, x, k):
        a = x[:, :, :, k, :]
        return torch.cat([a[..., :self.DEPTH_CH], a[..., self.DEPTH_CH + 1:]], -1).permute(0, 3, 1, 2)

    def _aux_encoder(self, x):
        # Covariates at t=1: T is 0 and the rain in force is the t=1 value.
        a = x[:, :, :, 0, :].clone()
        a[..., self.T_CH] = 0.0
        if self.physics:
            a[..., self.RAIN_T] = a[..., self.RAIN_1]
        return torch.cat([a[..., :self.DEPTH_CH], a[..., self.DEPTH_CH + 1:]], -1).permute(0, 3, 1, 2)

    def forward(self, x):
        B, h, w, K, _ = x.shape
        cur = x[:, :, :, 0, self.DEPTH_CH].unsqueeze(1)
        state = self.cell.init_state(B, (h, w), x.device, x.dtype)
        state = self.cell(torch.cat([cur, self._aux_encoder(x)], 1), state)
        outs = []
        for k in range(K):
            state = self.cell(torch.cat([cur, self._aux(x, k)], 1), state)
            cur = self.head(self.mid(state[0]))
            outs.append(cur)
        return torch.cat(outs, 1).permute(0, 2, 3, 1)


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def build_model(cfg, verbose=True):
    m = dict(cfg.model)
    model = RolloutConvLSTM(IN_CH, hidden=m["hidden"], kernel=m["kernel"], norm=m["norm"],
                            head_width=m["head_width"], physics=cfg.use_physics)
    if verbose:
        print(f"  {cfg.tag}: {n_params(model):,} parameters | ConvLSTM hidden {m['hidden']} | "
              f"{IN_CH} input channels | {cfg.out_steps}-step free-running rollout")
    return model.to(DEVICE)


MODEL_NOTE = "ConvLSTM (M1's cell and output head), 19-step free-running rollout from the t=1 field"
print(f"{TAG}: {MODEL_NOTE}")
_m = build_model(CFG)
del _m
torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

# Measured peak bytes per (H * W * K) cell of one full-domain no-grad rollout; decides GPU vs CPU evaluation.
EVAL_BYTES_PER_CELL = 240.0 * (HIDDEN / 64.0)
EVAL_FIXED_BYTES = 0.25e9

In [ ]:
DEPTH_CH = CHANNELS.index("depth_t1")


def rel_l2(pred, targ, denom=None):
    dims = tuple(range(1, targ.ndim))
    if denom is None:
        denom = torch.sqrt((targ ** 2).sum(dims))
    return (torch.sqrt(((pred - targ) ** 2).sum(dims)) / (denom + 1e-8)).mean()


def forward_depth(model, x, wet=None):
    # Always decoded with the training region's NORM, so transferred weights stay frozen.
    z = model(x)
    if CFG.residual:
        z = z + x[..., 0, DEPTH_CH].unsqueeze(-1)
    return decode_depth_t(z, NORM, wet)


def loss_fn(model, x, y_m, denom=None):
    if CFG.loss_space == "normalised":
        z_pred = model(x)
        if CFG.residual:
            z_pred = z_pred + x[..., 0, DEPTH_CH].unsqueeze(-1)
        z_targ = (y_m - NORM["depth"][0]) / NORM["depth"][1]
        return rel_l2(z_pred, z_targ)
    return rel_l2(forward_depth(model, x), y_m, denom)


# A crop is normalised by its block's full-domain ||y|| scaled to the crop's share of the domain, so the crop
# loss estimates the paper's full-domain Eq. 2 instead of re-weighting shallow crops.
Y_NORM = {}


def block_norm(ctx, b):
    key = (ctx["region"], int(b))
    if key not in Y_NORM:
        s = 0.0
        for f in range(int(b) * BLOCK + 1, (int(b) + 1) * BLOCK):
            a = np.asarray(ctx["depth"][f], np.float64)
            s += float((a * a).sum())
        Y_NORM[key] = math.sqrt(s)
    return Y_NORM[key]


def crop_denom(ctx, blocks, rows, cols, device):
    if rows is None:
        return None
    frac = (rows[1] - rows[0]) * (cols[1] - cols[0]) / float(ctx["H"] * ctx["W"])
    return torch.tensor([block_norm(ctx, b) * math.sqrt(frac) for b in blocks], dtype=torch.float32,
                        device=device)


def eval_device_for(ctx, note=True):
    # Decided in advance: on Windows an oversized CUDA pass pages to host RAM instead of raising OOM.
    if EVAL_DEVICE != "auto":
        return torch.device(EVAL_DEVICE)
    if DEVICE.type != "cuda":
        return DEVICE
    est = EVAL_BYTES_PER_CELL * ctx["H"] * ctx["W"] * OUT_STEPS + EVAL_FIXED_BYTES
    fits = est < EVAL_VRAM_FRAC * VRAM_BYTES
    if note:
        print(f"  {ctx['region']} {ctx['H']}x{ctx['W']}: full-domain pass needs ~{est / 1e9:.1f} GB of "
              f"{VRAM_BYTES / 1e9:.1f} GB -> {'cuda' if fits else 'cpu'}")
    return DEVICE if fits else torch.device("cpu")


@torch.no_grad()
def evaluate(model, ctx, blocks, norm, device=None, max_samples=None, keep=(), progress=False):
    device = device or eval_device_for(ctx)
    was = next(model.parameters()).device
    model.to(device).eval()
    blocks = list(blocks)[:max_samples] if max_samples else list(blocks)
    acc, acc_c, saved = PaperMetrics(), PaperMetrics(), {}
    t0 = time.time()
    for i, b in enumerate(blocks):
        x, y = make_sample(ctx, int(b), norm)
        xt, yt = to_torch(x, y, device)
        pred = forward_depth(model, xt)
        acc.update(pred.cpu(), yt.cpu())
        acc_c.update(pred.clamp(min=0.0).cpu(), yt.cpu())
        if int(b) in keep:
            saved[int(b)] = (y[..., -1].copy(), pred[0, :, :, -1].cpu().numpy())
        del xt, yt, pred
        if progress and (i + 1) % max(1, len(blocks) // 4) == 0:
            print(f"    {i + 1}/{len(blocks)} samples  {time.time() - t0:.0f}s", flush=True)
    model.to(was)
    if device.type == "cuda":
        torch.cuda.empty_cache()
    out = acc.result()
    out["clamped"] = {k: v for k, v in acc_c.result().items() if not k.startswith("_")}
    out["eval_device"] = device.type
    out["eval_seconds"] = time.time() - t0
    return (out, saved) if keep else out


_vrng = np.random.default_rng(12345)
VAL_WINDOWS = [(int(b), *sample_patch(CTX, _vrng))
               for b in (VAL_B[:VAL_SAMPLES] if VAL_SAMPLES else VAL_B)]


@torch.no_grad()
def quick_val(model):
    model.eval()
    tot = 0.0
    for b, rows, cols in VAL_WINDOWS:
        x, y = make_sample(CTX, b, NORM, rows, cols)
        xt, yt = to_torch(x, y, DEVICE)
        tot += rel_l2(forward_depth(model, xt), yt, crop_denom(CTX, [b], rows, cols, DEVICE)).item()
    model.train()
    return tot / len(VAL_WINDOWS)


print(f"loss: relative L2 (paper Eq. 2) in {CFG.loss_space} | residual head: {CFG.residual}"
      + (" | crops normalised by their sample's full-domain norm" if PATCH else ""))
print(f"validation during training: {len(VAL_WINDOWS)} fixed "
      + (f"{PATCH}x{PATCH} windows" if PATCH else "full-domain samples") + f", every {VAL_EVERY} epochs")
for _r in (REGION,) + TRANSFER:
    eval_device_for(load_region(_r, verbose=False))

In [ ]:
def train(cfg, verbose=True):
    set_seed(cfg.seed)
    model = build_model(cfg, verbose=verbose)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, betas=(0.9, 0.999),
                           weight_decay=cfg.weight_decay)
    steps_per_epoch = max(1, len(TRAIN_B) // cfg.batch_size)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs * steps_per_epoch, eta_min=0.0)
    rng = np.random.default_rng(cfg.seed)
    hist = {"epoch": [], "train_loss": [], "lr": [], "val_epoch": [], "val_rel_l2": []}
    best = {"val": float("inf"), "epoch": -1}
    t0 = time.time()
    model.train()
    for ep in range(1, cfg.epochs + 1):
        order = rng.permutation(TRAIN_B)
        run, n = 0.0, 0
        for s in range(steps_per_epoch):
            bs = order[s * cfg.batch_size:(s + 1) * cfg.batch_size]
            if len(bs) == 0:
                continue
            rows, cols = sample_patch(CTX, rng)
            xs, ys = zip(*[make_sample(CTX, int(b), NORM, rows, cols) for b in bs])
            xt = torch.from_numpy(np.stack(xs)).to(DEVICE)
            yt = torch.from_numpy(np.stack(ys)).to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model, xt, yt, crop_denom(CTX, bs, rows, cols, DEVICE))
            loss.backward()
            if cfg.grad_clip:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            sched.step()
            run += loss.item(); n += 1
            del xt, yt, loss
        hist["epoch"].append(ep); hist["train_loss"].append(run / max(n, 1))
        hist["lr"].append(sched.get_last_lr()[0])
        line = (f"  epoch {ep:3d}/{cfg.epochs}  train rel-L2 {run / max(n, 1):.6f}  "
                f"lr {sched.get_last_lr()[0]:.2e}  {fmt_hms(time.time() - t0)}")
        if ep % VAL_EVERY == 0 or ep == cfg.epochs:
            v = quick_val(model)
            hist["val_epoch"].append(ep); hist["val_rel_l2"].append(v)
            line += f"  |  val rel-L2 {v:.6f}"
            if v < best["val"]:
                best = {"val": v, "epoch": ep}
                # complete=False marks a mid-run checkpoint, which is never scored.
                torch.save({"state_dict": model.state_dict(), "epoch": ep, "val": v,
                            "config": asdict(cfg), "key": experiment_key(cfg),
                            "complete": False, "epochs_done": ep}, CKPT_PATH)
                line += "  <- best"
        if verbose:
            print(line, flush=True)
    hist["seconds"] = time.time() - t0
    hist["best"] = best
    hist["final_state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    global BEST_STATE
    BEST_STATE = None
    if best["epoch"] > 0:
        BEST_STATE = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)["state_dict"]
        print(f"  reporting the FINAL-epoch weights (val rel-L2 {hist['val_rel_l2'][-1]:.6f}); best validation was "
              f"epoch {best['epoch']} at {best['val']:.6f} and is scored as a secondary row")
    torch.save({"state_dict": model.state_dict(), "final_state": hist["final_state"], "best_state": BEST_STATE,
                "history": {k: v for k, v in hist.items() if k != "final_state"},
                "config": asdict(cfg), "key": experiment_key(cfg),
                "complete": True, "epochs_done": cfg.epochs}, CKPT_PATH)
    return model, hist


def load_checkpoint():
    if OVERWRITE or not os.path.exists(CKPT_PATH):
        return None
    try:
        ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
    except Exception as exc:
        print(f"  checkpoint {os.path.basename(CKPT_PATH)} is unreadable ({exc}); it will be retrained")
        return None
    if ck.get("key") != experiment_key(CFG):
        print(f"  checkpoint {os.path.basename(CKPT_PATH)} was trained under a DIFFERENT configuration; "
              f"it will be retrained")
        return None
    if not ck.get("complete"):
        print(f"  checkpoint {os.path.basename(CKPT_PATH)} stopped at epoch {ck.get('epochs_done')} of "
              f"{CFG.epochs} and is not a finished run; it will be retrained")
        return None
    return ck


BEST_STATE = None
EXISTING = None if OVERWRITE else load_json(RESULT_JSON)
TRAINED  = False
if EXISTING and EXISTING.get("key") == experiment_key(CFG) and EXISTING.get("complete") and RUN_MODE != "evaluate":
    print(f"this exact run is already on disk: {os.path.basename(RESULT_JSON)}")
    print(f"  RUN_MODE = 'evaluate' re-scores it from the checkpoint; OVERWRITE = True retrains from scratch.")
    RESULT, MODEL, HIST = EXISTING, None, EXISTING.get("history")
elif RUN_MODE == "check":
    print("RUN_MODE = 'check': caches, checks, the persistence reference and the parameter count only.")
    RESULT, MODEL, HIST = {}, None, None
else:
    _ck = load_checkpoint()
    if _ck is not None:
        MODEL = build_model(CFG, verbose=False)
        MODEL.load_state_dict(_ck["state_dict"])
        HIST = _ck.get("history") or {}
        BEST_STATE = _ck.get("best_state")
        print(f"scoring the trained weights in {os.path.basename(CKPT_PATH)} - no retraining "
              f"({_ck['epochs_done']} epochs, best val rel-L2 "
              f"{HIST.get('best', {}).get('val', float('nan')):.6f}).")
    elif RUN_MODE == "evaluate":
        raise FileNotFoundError(
            f"RUN_MODE = 'evaluate' needs a finished checkpoint at {CKPT_PATH}, and there is none for this "
            f"configuration. Run with RUN_MODE = 'train' first.")
    else:
        print(f"\ntraining {TAG} on {REGION}: {len(TRAIN_B)} samples x {CFG.epochs} epochs, "
              f"batch {CFG.batch_size}"
              + (f", random {PATCH}x{PATCH} crops" if PATCH else ", full domain"))
        MODEL, HIST = train(CFG)
        TRAINED = True
        print(f"\ntrained in {fmt_hms(HIST['seconds'])}")
    RESULT = {}


def save_result(**parts):
    RESULT.update(parts)
    RESULT["saved"] = time.strftime("%Y-%m-%d %H:%M:%S")
    save_json(RESULT_JSON, RESULT)
    return RESULT

In [ ]:
if MODEL is not None:
    print("scoring on the full domain:")
    VAL_M  = evaluate(MODEL, CTX, VAL_B, NORM)
    TEST_M, KEEP = evaluate(MODEL, CTX, TEST_B, NORM, keep=(int(TEST_B[0]), int(TEST_B[-1])), progress=True)
    TEST_BEST = None
    if BEST_STATE is not None:
        _bm = build_model(CFG, verbose=False)
        _bm.load_state_dict(BEST_STATE)
        TEST_BEST = evaluate(_bm, CTX, TEST_B, NORM)
        del _bm
    save_result(tag=TAG, region=REGION, seed=SEED,
                protocol="FloodCastBench paper protocol (Xu et al. 2025, Sci Data 12:431): non-overlapping "
                         "blocks of 20 frames at 300 s, predict t=2..20 from t=1",
                model_note=MODEL_NOTE, input_channels=CHANNELS, n_parameters=int(n_params(MODEL)),
                config=asdict(CFG), key=experiment_key(CFG),
                splits={"train": len(TRAIN_B), "val": len(VAL_B), "test": len(TEST_B)},
                norm=dict(NORM), test=TEST_M, val=VAL_M, test_best_val_weights=TEST_BEST,
                reported_weights="final epoch",
                persistence_test=REF_TEST, persistence_val=REF_VAL,
                paper_table4=paper_rows(PAPER_TABLE4, REGION), complete=False)
    print(f"  metrics written to {os.path.basename(RESULT_JSON)}")
    rows = {TAG + " (final)": TEST_M, **({TAG + " (best val)": TEST_BEST} if TEST_BEST else {}),
            "persistence": REF_TEST, **paper_rows(PAPER_TABLE4, REGION)}
    results_table(rows, title=f"\n{REGION} TEST ({len(TEST_B)} samples, 19 steps = 95 min lead)"
                              f"   [* = best of the rows measured here]\n")
    print(f"\n  val rel-L2 {VAL_M['RMSE']:.6f}  vs  test rel-L2 {TEST_M['RMSE']:.6f}"
          f"   | persistence: val {REF_VAL['RMSE']:.6f}, test {REF_TEST['RMSE']:.6f}")
    if TEST_M["RMSE"] >= REF_TEST["RMSE"]:
        print("  WARNING: the model does not beat persistence on the test split.")
    if abs(VAL_M["RMSE"] - TEST_M["RMSE"]) > 3 * VAL_M["RMSE"]:
        print("  WARNING: validation and test differ by more than 3x.")
else:
    VAL_M = TEST_M = KEEP = None

In [ ]:
TRANSFER_M = {}
if MODEL is not None and TRANSFER:
    for tgt in TRANSFER:
        tctx = load_region(tgt)
        tnorm = NORM
        if TRANSFER_RENORM:
            tnorm = fit_norm(tctx, np.arange(tctx["n_blocks"]))
        all_b = np.arange(tctx["n_blocks"])
        m, kp = evaluate(MODEL, tctx, all_b, tnorm, keep=(int(all_b[-1]),), progress=True)
        TRANSFER_M[tgt] = m
        TRANSFER_M[tgt]["_keep"] = kp
        save_result(transfer={k: {kk: vv for kk, vv in v.items() if kk != "_keep"}
                              for k, v in TRANSFER_M.items()},
                    transfer_renormalised=TRANSFER_RENORM,
                    paper_table5={t: paper_rows(PAPER_TABLE5, t) for t in TRANSFER})
        ref = persistence_reference(tctx, all_b)
        results_table({f"{TAG} (frozen)": m, "persistence": ref, **paper_rows(PAPER_TABLE5, tgt)},
                      title=f"\n{REGION} -> {tgt}, zero-shot, {len(all_b)} samples"
                            f"{' [target-renormalised inputs]' if TRANSFER_RENORM else ''}\n")
        print(f"\n  Grid: {CTX['H']}x{CTX['W']} at ~{CTX['spec']['cell_m']:.0f} m  ->  "
              f"{tctx['H']}x{tctx['W']} at ~{tctx['spec']['cell_m']:.0f} m")
elif MODEL is not None:
    print(f"no transfer target for {REGION}")

In [ ]:
if MODEL is not None:
    fig = plt.figure(figsize=(16, 9))
    gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.25])

    _h = HIST or {}
    ax = fig.add_subplot(gs[0, 0])
    if _h.get("epoch"):
        ax.plot(_h["epoch"], _h["train_loss"], label="train")
        ax.plot(_h["val_epoch"], _h["val_rel_l2"], "o-", ms=3, label="val")
        ax.set(yscale="log"); ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "no training history\nin the checkpoint", ha="center", va="center", fontsize=9)
    ax.set(xlabel="epoch", ylabel="relative L2", title=f"{TAG}: training")
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 1])
    if _h.get("epoch"):
        ax.plot(_h["epoch"], _h["lr"])
    ax.set(xlabel="epoch", ylabel="lr", title="cosine schedule, 1e-3 -> 0")
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 2])
    lead = (np.arange(OUT_STEPS) + 2) * 300 / 60.0
    ax.plot(lead, TEST_M["_per_step_RMSE_m"], label=TAG)
    ax.plot(lead, REF_TEST["_per_step_RMSE_m"], "--", label="persistence")
    ax.set(xlabel="lead [min]", ylabel="RMSE [m]", title="error against lead time, TEST")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)

    ax = fig.add_subplot(gs[0, 3]); ax.axis("off")
    ax.text(0.0, 0.95, f"{TAG}  |  {REGION}  |  seed {SEED}", fontsize=11, weight="bold", va="top")
    _lines = [f"{MODEL_NOTE}", "",
              f"inputs      {', '.join(CHANNELS)}",
              f"parameters  {n_params(MODEL):,}",
              f"train       {len(TRAIN_B)} samples x {CFG.epochs} epochs, batch {CFG.batch_size}",
              f"geometry    " + (f"{PATCH}x{PATCH} crops" if PATCH else "full domain"),
              f"wall clock  {fmt_hms(HIST['seconds'])}", "",
              f"TEST rel-L2 {TEST_M['RMSE']:.6f}   (persistence {REF_TEST['RMSE']:.6f})",
              f"TEST NSE    {TEST_M['NSE']:.6f}",
              f"TEST CSI    {TEST_M['CSI@0.001']:.4f} / {TEST_M['CSI@0.01']:.4f}  (g = 1 mm / 1 cm)"]
    ax.text(0.0, 0.85, "\n".join(_lines), fontsize=8.5, va="top", family="monospace")

    _b = int(TEST_B[0])
    _truth, _pred = KEEP[_b]
    # 99.5th-percentile colour scale: a few deep river cells would otherwise wash out the floodplain.
    _vmax = max(float(np.percentile(_truth, 99.5)), 1e-3)
    _err  = np.abs(_pred - _truth)
    for _i, (_a, _t, _vm) in enumerate(((_truth, f"{REGION} TEST truth, t=20", _vmax),
                                        (_pred, f"{TAG} prediction, t=20", _vmax),
                                        (_err, "|error| [m]", max(float(np.percentile(_err, 99.5)), 1e-4)))):
        _ax2 = fig.add_subplot(gs[1, _i])
        _im = _ax2.imshow(_a, cmap="Blues" if _i < 2 else "magma", vmin=0, vmax=_vm)
        _ax2.set_title(_t + ("  (99.5th pct scale)" if _i == 0 else ""), fontsize=9)
        _ax2.axis("off")
        plt.colorbar(_im, ax=_ax2, fraction=0.046, extend="max")
    _ax2 = fig.add_subplot(gs[1, 3])
    if TRANSFER_M:
        _tgt = list(TRANSFER_M)[0]
        _kb = list(TRANSFER_M[_tgt]["_keep"])[0]
        _tt, _tp = TRANSFER_M[_tgt]["_keep"][_kb]
        _te = np.abs(_tp - _tt)
        _im = _ax2.imshow(_te, cmap="magma", vmin=0, vmax=max(float(np.percentile(_te, 99.5)), 1e-4))
        _ax2.set_title(f"{_tgt} frozen transfer, |error| t=20", fontsize=9)
        plt.colorbar(_im, ax=_ax2, fraction=0.046, extend="max")
    _ax2.axis("off")
    plt.suptitle(f"{TAG} on FloodCastBench, paper protocol (Xu et al. 2025): t=1 -> t=2..20, 95 min lead",
                 fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{RUN_NAME}_summary.png"), dpi=110, bbox_inches="tight")
    plt.show()

In [ ]:
if MODEL is not None:
    for _t in TRANSFER_M.values():
        _t.pop("_keep", None)
    save_result(transfer=TRANSFER_M, transfer_renormalised=TRANSFER_RENORM,
                paper_table5={t: paper_rows(PAPER_TABLE5, t) for t in TRANSFER},
                history={k: v for k, v in (HIST or {}).items() if k != "final_state"},
                wall_clock_seconds=(HIST or {}).get("seconds"),
                trained_this_session=TRAINED, complete=True)
    print(f"saved {os.path.basename(RESULT_JSON)} and {RUN_NAME}_summary.png in {RESULTS_DIR}")
    print(f"  weights are in {os.path.basename(CKPT_PATH)}"
          + (" (written when training finished)" if TRAINED else " (unchanged; this run only scored them)"))

if RESULT:
    print(f"\n{'=' * 100}")
    print(f"{TAG} | {REGION} | seed {SEED}   TEST  rel-L2 {RESULT['test']['RMSE']:.6f}   "
          f"NSE {RESULT['test']['NSE']:.6f}   r {RESULT['test']['r']:.6f}   "
          f"CSI {RESULT['test']['CSI@0.001']:.4f} / {RESULT['test']['CSI@0.01']:.4f}")
    for _t, _m in (RESULT.get("transfer") or {}).items():
        print(f"{'':>{len(TAG)}}   -> {_t:12s} frozen   rel-L2 {_m['RMSE']:.6f}   NSE {_m['NSE']:.6f}   "
              f"r {_m['r']:.6f}   CSI {_m['CSI@0.001']:.4f} / {_m['CSI@0.01']:.4f}")
    print(f"{'=' * 100}")